In [2]:
import numpy as np
import torch


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/mac/anaconda3/lib/python3.11/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/Users/mac/anaconda3/lib/python3.11/site-packages/traitlets/config/application.py", line 992, in launch_instance
    app.start()
  File "/Users/mac/anaconda3/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 736, in start
    self.io_loop.start()
  File "/Users/mac/anacond

In [4]:


# Default value ; unit: m
D_width = 0.1
D_height = 0.2
D_young_modulus = 210e9
D_shear_modulus = 81e9
D_poisson_ratio = 0.3
n_dof_per_node = 6
cross_section_angle_a = 0  # Anti-clockwise positiveness
cross_section_angle_b = 0


def rotation(v, k, theta):
    # V is the vector been rotated; k is the axis about; theta is the angle
    k = k / torch.norm(k)  # Normalizing the axis k
    cross_product = torch.cross(k, v)
    dot_product = torch.dot(k, v)
    v_rotated = v * torch.cos(theta) + cross_product * torch.sin(theta) + k * dot_product * (1 - torch.cos(theta))

    return v_rotated


class Beam:
    def __init__(self, node_coordinates, width=D_width, height=D_height, young_modulus=D_young_modulus,
                 shear_modulus=D_shear_modulus, poisson_ratio=D_poisson_ratio, Beta_a=cross_section_angle_a,
                 Beta_b=cross_section_angle_b):
        # Coordinates tensor, shaped as (2, 3)
        self.node_coordinates = node_coordinates

        # Material
        self.width = width
        self.height = height
        self.young_modulus = young_modulus
        self.shear_modulus = shear_modulus
        self.poisson_ratio = poisson_ratio

        # Moment of inertia
        self.length = torch.norm(self.node_coordinates[1] - self.node_coordinates[0])  # 计算两个节点之间的长度
        self.Iy = (self.width * self.height ** 3) / 12
        self.Iz = (self.width ** 3 * self.height) / 12
        self.A = self.width * self.height
        self.J = (self.width * self.height ** 3) / 3

        # Stiffness components
        self.S_u = self.young_modulus * self.A / self.length
        self.S_v1a = 12 * self.young_modulus * self.Iy / (self.length ** 3)
        self.S_v1b = 6 * self.young_modulus * self.Iy / (self.length ** 2)
        self.S_v2a = 12 * self.young_modulus * self.Iz / (self.length ** 3)
        self.S_v2b = 6 * self.young_modulus * self.Iz / (self.length ** 2)
        self.S_theta1a = 6 * self.young_modulus * self.Iy / (self.length ** 2)
        self.S_theta1b = 4 * self.young_modulus * self.Iy / self.length
        self.S_theta1c = 2 * self.young_modulus * self.Iy / self.length
        self.S_theta2a = 6 * self.young_modulus * self.Iz / (self.length ** 2)
        self.S_theta2b = 4 * self.young_modulus * self.Iz / self.length
        self.S_theta2c = 2 * self.young_modulus * self.Iz / self.length
        self.S_Tr = self.shear_modulus * self.J / self.length

        # Extras
        # Cross-sectional Rotation at the two ends
        self.Beta_a = Beta_a
        self.Beta_b = Beta_b

    def get_element_stiffness_matrix(self):
        """
        Stiffness matrix at element level
        """
        K_element = torch.tensor([
            [self.S_u, 0, 0, 0, 0, 0, -self.S_u, 0, 0, 0, 0, 0],
            [0, self.S_v1a, 0, 0, 0, self.S_theta1a, 0, -self.S_v1a, 0, 0, 0, self.S_theta1a],
            [0, 0, self.S_v2a, 0, -self.S_theta2a, 0, 0, 0, -self.S_v2a, 0, -self.S_theta2a, 0],
            [0, 0, 0, self.S_Tr, 0, 0, 0, 0, 0, -self.S_Tr, 0, 0],
            [0, 0, -self.S_v2b, 0, self.S_theta2b, 0, 0, 0, self.S_v2b, 0, self.S_theta2c, 0],
            [0, self.S_v1b, 0, 0, 0, self.S_theta1b, 0, -self.S_v1b, 0, 0, 0, self.S_theta1c],
            [-self.S_u, 0, 0, 0, 0, 0, self.S_u, 0, 0, 0, 0, 0],
            [0, -self.S_v1a, 0, 0, 0, -self.S_theta1a, 0, self.S_v1a, 0, 0, 0, -self.S_theta1a],
            [0, 0, -self.S_v2a, 0, self.S_theta2a, 0, 0, 0, self.S_v2a, 0, self.S_theta2a, 0],
            [0, 0, 0, -self.S_Tr, 0, 0, 0, 0, 0, self.S_Tr, 0, 0],
            [0, 0, -self.S_v2b, 0, self.S_theta2c, 0, 0, 0, self.S_v2b, 0, self.S_theta2b, 0],
            [0, self.S_v1b, 0, 0, 0, self.S_theta1c, 0, -self.S_v1b, 0, 0, 0, self.S_theta1b],
        ], dtype=torch.float32)

        return K_element

    def System_Transform(self):
        """
        Coordinates transformation
        """
        vector_x = self.node_coordinates[1, 0] - self.node_coordinates[0, 0]
        vector_y = self.node_coordinates[1, 1] - self.node_coordinates[0, 1]
        vector_z = self.node_coordinates[1, 2] - self.node_coordinates[0, 2]
        length = torch.norm(self.node_coordinates[1] - self.node_coordinates[0])

        # Calculate alpha and ceta using PyTorch
        alpha = torch.acos(vector_x / torch.sqrt(vector_y ** 2 + vector_x ** 2))
        ceta = torch.acos(vector_z / length)

        Projection_Z_x = vector_z * length * torch.cos(-(torch.pi / 2 + alpha))
        Projection_Z_y = vector_z * length * torch.sin(-alpha)
        Projection_Z_z = torch.sin(ceta)

        V_projection = torch.stack([Projection_Z_x, Projection_Z_y, Projection_Z_z])
        X_axis = torch.tensor([vector_x / length, vector_x / length, vector_z / length])
        Z_axis_a = rotation(V_projection, X_axis, self.Beta_a)
        Z_axis_b = rotation(V_projection, X_axis, self.Beta_b)
        Y_axis_a = rotation(Z_axis_a, X_axis, torch.pi / 2)
        Y_axis_b = rotation(Z_axis_b, X_axis, torch.pi / 2)
        Z_axis_a = Z_axis_a / torch.norm(Z_axis_a)
        Z_axis_b = Z_axis_b / torch.norm(Z_axis_b)
        Y_axis_a = Y_axis_a / torch.norm(Y_axis_a)
        Y_axis_b = Y_axis_b / torch.norm(Y_axis_b)

        lambda_matrix = torch.tensor([
            [X_axis[0], X_axis[1], X_axis[2]],
            [Y_axis_a[0], Y_axis_a[1], Y_axis_a[2]],
            [Z_axis_a[0], Z_axis_a[1], Z_axis_a[2]]
        ], dtype=torch.float32)

        matrix_T = torch.zeros((18, 18), dtype=torch.float32)
        for i in range(0, 18, 3):
            matrix_T[i:i + 3, i:i + 3] = lambda_matrix

        return matrix_T


def assemble_stiffness_matrix(beams, n_elements, n_dof_per_node):
    """
    Stiffness matrix at global level
    """
    total_dof = (n_elements + 1) * n_dof_per_node  # 总自由度数
    K_global = torch.zeros((total_dof, total_dof), dtype=torch.float32)

    for i in range(n_elements):
        Matrix_T = beams[i].System_Transform()  # 获取变换矩阵
        K_element = torch.matmul(torch.transpose(Matrix_T, 0, 1), torch.matmul(beams[i].get_element_stiffness_matrix(), Matrix_T))

        start_idx = i * n_dof_per_node
        end_idx = start_idx + 12

        K_global[start_idx:end_idx, start_idx:end_idx] += K_element

    return K_global


# Example usage
node_coords_1 = torch.tensor([[0.0, 0.0, 0.0], [1.0, 0.0, 0.0]], dtype=torch.float32)
node_coords_2 = torch.tensor([[1.0, 0.0, 0.0], [2.0, 0.0, 0.0]], dtype=torch.float32)

beam_1 = Beam(node_coordinates=node_coords_1, width=D_width, height=D_height,
              young_modulus=D_young_modulus, shear_modulus=D_shear_modulus, poisson_ratio=D_poisson_ratio)
beam_2 = Beam(node_coordinates=node_coords_2, width=D_width, height=D_height,
              young_modulus=D_young_modulus, shear_modulus=D_shear_modulus, poisson_ratio=D_poisson_ratio)

beams = [beam_1, beam_2]

n_elements = 2
K_global = assemble_stiffness_matrix(beams, n_elements, n_dof_per_node)


# BCs (fixed node1)
fixed_dof = [0, 1, 2, 3, 4, 5]  # fixed Dof index

# Project in the stiffness matrix
K_global[fixed_dof, :] = 0
K_global[:, fixed_dof] = 0
K_global[fixed_dof, fixed_dof] = 1e5
# print(K)

total_dof = (n_elements + 1) * 6 
# External Force
F = torch.zeros(total_dof, dtype=torch.float32)
F[12] = 1000  # Suppose a transverse force of 1000N at the node 2

# Displacement solution
rank_K = torch.linalg.matrix_rank(K_global)
print(rank_K)
print(K_global.shape)
# Check mathematical viability
if rank_K < K_global.shape[0]:
    print("Warning: The stiffness matrix is not of full rank")
else:
    displacements = torch.linalg.solve(K_global, F)
    print("Displacement vector:", displacements)
    reaction_forces = torch.matmul(K_global, displacements)
    print("Internal force vector:", reaction_forces)


/var/folders/0j/34jt056n1qd03z4ldfbf3d5r0000gn/T/ipykernel_45717/3692022487.py:15: UserWarning: Using torch.cross without specifying the dim arg is deprecated.
Please either pass the dim explicitly or simply use torch.linalg.cross.
The default value of dim will change to agree with that of linalg.cross in a future release. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/Cross.cpp:67.)
  cross_product = torch.cross(k, v)


TypeError: cos(): argument 'input' (position 1) must be Tensor, not int